# 09 · Convolution and deconvolution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/09-convolution-and-deconvolution.ipynb)

*Part IV · exercise · 15 min*

> 🇪🇸 **Convolución y deconvolución** — La convolución es un producto matricial estructurado, y el desenfoque se puede deshacer en parte.

Convolution is a structured matrix product, and blur can be partly undone.

## What you will be able to do

- Predict the output size of a convolution in `full`, `valid` and `same` mode.
- Say why what deep learning calls convolution is really correlation.
- Write a convolution as multiplication by a Toeplitz matrix.
- Distinguish transposed convolution from true deconvolution.
- Recover a blurred image with Richardson-Lucy, and measure it honestly.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
from scipy import signal
from scipy.linalg import toeplitz
from skimage import data
from skimage.restoration import richardson_lucy

img = data.camera().astype(float) / 255.      # real photograph, 512x512
sobel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
print(img.shape, img.min(), img.max())

## The theory

> 🇪🇸 La convolución desliza un núcleo pequeño sobre un array mayor,
> multiplicando y sumando en cada posición.

**Convolution** slides a small array (the **kernel**, or **filter**) across a
larger one, multiplying and summing at each position. It is the operation at the
heart of every convolutional neural network, and it is also how every blur,
sharpen and edge-detection filter works.

In [ ]:
x = np.array([1., 2., 3., 4., 5.])
k = np.array([1., 0., -1.])

print(np.convolve(x, k, 'full'))    # [ 1.  2.  2.  2.  2. -4. -5.]  length 5+3-1 = 7
print(np.convolve(x, k, 'valid'))   # [ 2.  2.  2.]                  length 5-3+1 = 3
print(np.convolve(x, k, 'same'))    # [ 2.  2.  2.  2. -4.]          length 5

Three modes, three output sizes. `valid` uses only positions where the kernel
fits completely — this is why convolution **shrinks** an image by
`kernel_size - 1`.

⚠️ **A detail that confuses everyone.** True convolution flips the kernel;
**correlation** does not. What deep learning libraries call "convolution" is
actually correlation. It makes no practical difference, because the network
*learns* the kernel — but you should know the names are inconsistent.

In [ ]:
print(np.correlate(x, k, 'valid'))          # [-2. -2. -2.]
print(np.convolve(x, k[::-1], 'valid'))     # [-2. -2. -2.] — the same, with k flipped

### Convolution is a matrix multiplication

This is the connection back to Chapter 2. Any convolution can be written as
multiplication by a **Toeplitz** matrix — a matrix where the kernel is shifted
along each row.

In [ ]:
col = np.zeros(7); col[:3] = k
row = np.zeros(5); row[0] = k[0]
C = toeplitz(col, row)                             # (7, 5)
print(C.shape)
print(np.allclose(C @ x, np.convolve(x, k, 'full')))   # True

So convolution is not a new kind of operation. It is a **structured matrix
multiplication** — one where the same few numbers are reused across the whole
matrix. That reuse is exactly why CNNs need so many fewer parameters than fully
connected networks.

### Deconvolution means two different things

Keep them separate:

1. **Transposed convolution** — the upsampling layer in a decoder or GAN. It
   makes things *bigger*. It is **not** a true inverse; the name is historical
   and misleading.
2. **True deconvolution** — recovering the original signal from a blurred one.
   This is a genuine inverse problem, and it is where section 07 comes back.

## Exercise 1 — convolve and blur a real photograph

> 🇪🇸 Convoluciona y desenfoca una fotografía real.

In [ ]:
# TODO 1: Convolve `img` with `sobel` in 'valid' mode. What shape comes out,
#         and by how much did it shrink?

# TODO 2: Blur the image with a 9x9 averaging kernel (all entries equal,
#         summing to 1), mode='same'. Display it next to the original.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
edges = signal.convolve2d(img, sobel, mode='valid')     # (510, 510) — shrank by 2
print(edges.shape)

psf = np.ones((9, 9)); psf /= psf.sum()
blurred = signal.convolve2d(img, psf, mode='same', boundary='symm')
print(blurred.shape)                                    # (512, 512) — 'same' keeps it

# import matplotlib.pyplot as plt
# fig, ax = plt.subplots(1, 3, figsize=(12, 4))
# for a, im, t in zip(ax, [img, edges, blurred], ["original", "sobel", "blurred"]):
#     a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")

# 'valid' shrinks by kernel_size - 1 = 2 in each direction: 512 -> 510.

## Exercise 2 — transposed convolution

> 🇪🇸 Convolución transpuesta: hace las cosas más grandes, no las deshace.

In [ ]:
# TODO 3: Upsample this 2x2 array to 3x3 by adding small * kernel into an
#         output array at each position:
#             small = np.array([[1., 2.], [3., 4.]]); ker = np.ones((2, 2))
#         What shape do you get? Why is this called "deconvolution" in CNNs
#         even though it does not undo anything?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
small = np.array([[1., 2.], [3., 4.]])
ker = np.ones((2, 2))

out = np.zeros((3, 3))
for i in range(2):
    for j in range(2):
        out[i:i+2, j:j+2] += small[i, j] * ker
print(out)
# [[ 1.  3.  2.]
#  [ 4. 10.  6.]
#  [ 3.  7.  4.]]

# Shape (3, 3): it GREW, by kernel_size - 1, which is exactly what 'valid'
# convolution shrinks by. That inverse relationship between the SHAPES is the
# whole reason for the name. The VALUES are not undone at all — you cannot
# recover `small` from `out`. The name is historical and misleading.

## Exercise 3 — true deconvolution

> 🇪🇸 Deconvolución de verdad. **Recorta 25 píxeles del borde antes de medir.**

::: {.callout-warning}
**This is the hardest thing in the workshop, and it has a trap.** Deconvolution
creates strong artifacts at the edges, where the algorithm has no information
about what lies outside the image. If you measure error over the whole image,
the artifacts dominate and it looks like the method failed. **Crop 25 pixels off
every side before measuring.** Do this before you run it, not after.
:::

In [ ]:
# TODO 4: Add small noise to the blurred image, then try to recover the
#         original with skimage.restoration.richardson_lucy(..., num_iter=50).
#         Measure error BEFORE and AFTER, ignoring a 25-pixel border.
#         Did it improve?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
psf = np.ones((9, 9)); psf /= psf.sum()
blurred = signal.convolve2d(img, psf, mode='same', boundary='symm')
noisy = blurred + 0.002 * np.random.default_rng(0).standard_normal(blurred.shape)

recovered = richardson_lucy(np.clip(noisy, 0, 1), psf, num_iter=50)

c = 25   # ignore the border: deconvolution always creates edge artifacts
err = lambda a: np.linalg.norm((a - img)[c:-c, c:-c]) / np.linalg.norm(img[c:-c, c:-c])
print(err(noisy), err(recovered))     # 0.1157 -> 0.0815

# Deconvolution REDUCED THE ERROR BY ABOUT 30%.

## Why not just invert the blur directly?

Because it fails badly. Blurring destroys high-frequency detail, so inverting it
divides by numbers very close to zero and amplifies noise enormously.

In [ ]:
K = np.fft.fft2(psf, s=img.shape)
naive = np.real(np.fft.ifft2(np.fft.fft2(noisy) / np.where(abs(K) < 1e-3, 1e-3, K)))
c = 25
err = lambda a: np.linalg.norm((a - img)[c:-c, c:-c]) / np.linalg.norm(img[c:-c, c:-c])
print(err(noisy), err(recovered), err(naive))
# 0.1157   0.0815   ~2.49
#
# The naive inverse is roughly TWENTY TIMES WORSE than the blurred image it
# started from. Cropping the border does not rescue it either (~2.51 uncropped):
# this is not an edge artifact, it is noise amplified across the whole image.

## What just happened

**This is the same lesson as section 07.** A direct inverse either does not exist
or is unusable, so you use a method that finds the best stable answer instead.
The pseudoinverse does this for linear systems; Richardson-Lucy and Wiener
filtering do it for deconvolution.

> 🇪🇸 Cuando no existe una inversa exacta, no te rindes: buscas la mejor
> aproximación estable.

In biotech this is routine: every fluorescence microscope blurs its images by a
known amount (the *point spread function*), and deconvolution is standard
practice before cells are counted or measured.

---

## Done with this section

Next up: **10 · Tucker decomposition on real data** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)